In [1]:
library(limma)
library(dplyr)
library(stringr)
library(data.table)
library(glue)
library(tidyverse)

Warning message:
"package 'dplyr' was built under R version 4.5.2"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'data.table' was built under R version 4.5.2"

Attaching package: 'data.table'


The following objects are masked from 'package:dplyr':

    between, first, last


Warning message:
"package 'ggplot2' was built under R version 4.5.2"
Warning message:
"package 'tibble' was built under R version 4.5.2"
Warning message:
"package 'tidyr' was built under R version 4.5.2"
Warning message:
"package 'readr' was built under R version 4.5.2"
Warning message:
"package 'purrr' was built under R version 4.5.2"
Warning message:
"package 'lubridate' was built under R version 4.5.2"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v forcats   1.0.1     v readr     2.2.0
v ggplot2   

In [2]:
output = "./output/limma/"

In [3]:
sample_info <- read.delim("input/sample_info_completed.tsv", check.names = FALSE)
sample_info <- sample_info[, c(
  "Sample ID",
  "TP",
  "Sex",
  "CF ID",
  "cancer_type",
  "stage",
  "toxicity_first_date",
  "toxicity",
  "country",
  "PPI",
  "Cortisone",
  "NSAID",
  "Asprin",
  "AB"
)]

meta <- read.delim("input/meta.tsv", check.names = FALSE)

quant <- read.delim("input/plasma_prot_quant.tsv", check.names = FALSE)
quant$PG.Genes <- ifelse(
  grepl(";", quant$PG.Genes),
  paste0(sub(";.*", "", quant$PG.Genes), " genes"),
  quant$PG.Genes
)

In [4]:
mapping <- setNames(meta$`Sample ID`, meta$R.FileName)
quant_cols <- colnames(quant)[colnames(quant) != "PG.Genes"]

col_to_cf <- list()

for (col in quant_cols) {
  match_idx <- which(sapply(names(mapping), function(rfile) grepl(rfile, col)))

  if (length(match_idx) > 0) {
    rfile <- names(mapping)[match_idx[1]]
    col_to_cf[[col]] <- mapping[[rfile]]
  }
}

mapped_cols <- names(col_to_cf)

quant_filtered <- quant[, c("PG.Genes", mapped_cols)]

colnames(quant_filtered)[match(mapped_cols, colnames(quant_filtered))] <- 
  unlist(col_to_cf)

quant_filtered <- quant_filtered[!duplicated(quant_filtered$PG.Genes), ]
rownames(quant_filtered) <- quant_filtered$PG.Genes
quant_filtered$PG.Genes <- NULL

quant_filtered <- as.matrix(quant_filtered)
mode(quant_filtered) <- "numeric"

Warning message in mde(x):
"NAs introduced by coercion"


In [5]:

# ---------------------------
# 1. Transpose (samples as rows, proteins as columns)
# ---------------------------
expr <- t(quant_filtered)

# ---------------------------
# 2. Ensure numeric matrix
# ---------------------------
expr <- apply(expr, 2, function(x) as.numeric(as.character(x)))
expr <- as.matrix(expr)
rownames(expr) <- colnames(quant_filtered)

# ---------------------------
# 3. Remove duplicated samples (rows)
# ---------------------------
expr <- expr[!duplicated(rownames(expr)), , drop = FALSE]

# ---------------------------
# 4. Log2 transform (log2(x + 1))
# ---------------------------
expr <- log2(expr + 1)

# ---------------------------
# 5. Ensure column names are character
# ---------------------------
colnames(expr) <- as.character(colnames(expr))

# ---------------------------
# 6. Remove duplicated protein columns
# ---------------------------
expr <- expr[, !duplicated(colnames(expr)), drop = FALSE]

In [14]:
library(limma)

limma_differential <- function(expr,
                               sample_info,
                               contrast_feature,
                               contrast_direction = "M_vs_F",
                               confounding_factors = c("country", "cancer_type", "TP")) {

  # ---------------------------
  # CLEAN FACTORS
  # ---------------------------
  clean_factor <- function(x) {
    x <- trimws(x)
    x[x == ""] <- NA
    factor(x)
  }

  sample_info[[contrast_feature]] <- clean_factor(sample_info[[contrast_feature]])
  sample_info <- sample_info[!is.na(sample_info[[contrast_feature]]), ]

  # Remove duplicates
  sample_info <- sample_info[!duplicated(sample_info$`CF ID`), ]
  expr <- expr[!duplicated(rownames(expr)), ]

  # ---------------------------
  # ALIGN DATA
  # ---------------------------
  common <- intersect(rownames(expr), sample_info$`CF ID`)

  expr <- expr[common, , drop = FALSE]
  sample_info <- sample_info[match(common, sample_info$`CF ID`), , drop = FALSE]

  rownames(sample_info) <- sample_info$`CF ID`

  # transpose for limma
  expr <- t(expr)

  # ---------------------------
  # ENSURE FACTORS
  # ---------------------------
  sample_info[[contrast_feature]] <- factor(sample_info[[contrast_feature]])

  for (f in confounding_factors) {
    sample_info[[f]] <- factor(sample_info[[f]])
  }

  # ---------------------------
  # PARSE CONTRAST
  # ---------------------------
  parts <- strsplit(contrast_direction, "_vs_")[[1]]

  if (length(parts) != 2) {
    stop("contrast_direction must be 'A_vs_B'")
  }

  group1 <- gsub("\\s+", "", parts[1])
  group2 <- gsub("\\s+", "", parts[2])

  print(glue("Group 1: {group1}"))
  print(glue("Group 2: {group2}"))

  # ---------------------------
  # FILTER TO CONTRAST LEVELS
  # ---------------------------

  sample_info[[contrast_feature]] <- trimws(sample_info[[contrast_feature]])
  # keep only requested groups
  keep_idx <- sample_info[[contrast_feature]] %in% c(group1, group2)
  if (sum(keep_idx) == 0) {
    stop("No samples left after filtering for contrast levels")
  }

  sample_info <- sample_info[keep_idx, , drop = FALSE]

  # Align expr as well  
  common <- intersect(colnames(expr), sample_info$`CF ID`)
  expr <- expr[, common, drop = FALSE]
  sample_info <- sample_info[match(common, sample_info$`CF ID`), , drop = FALSE]

  # ---------------------------
  # CHECK BINARY CONTRAST
  # ---------------------------
  if (length(unique(sample_info[[contrast_feature]])) != 2) {
    stop("Contrast feature must have exactly 2 levels.")
  }

  # ---------------------------
  # DESIGN MATRIX
  # ---------------------------
  design_formula <- as.formula(
    paste("~ 0 +", contrast_feature, "+",
          paste(confounding_factors, collapse = " + "))
  )

  design <- model.matrix(design_formula, data = sample_info)

  # ---------------------------
  # FIT MODEL
  # ---------------------------
  fit <- lmFit(expr, design)

  # ---------------------------
  # BUILD CONTRAST
  # ---------------------------
  group1_col <- paste0(contrast_feature, group1)
  group2_col <- paste0(contrast_feature, group2)

  if (!(group1_col %in% colnames(design)) |
      !(group2_col %in% colnames(design))) {
    stop("Contrast groups not found in design matrix")
  }

  contrast_name <- paste0(group1_col, " - ", group2_col)
  print(glue("Contrast: {contrast_name}"))

  contrast_matrix <- makeContrasts(
    contrasts = contrast_name,
    levels = design
  )

  # ---------------------------
  # EBAYES
  # ---------------------------
  fit2 <- contrasts.fit(fit, contrast_matrix)
  fit2 <- eBayes(fit2)

  # ---------------------------
  # RESULTS
  # ---------------------------
  res <- topTable(fit2,
                  number = Inf,
                  adjust.method = "BH",
                  sort.by = "P")

  res$Protein <- rownames(res)

  colnames(res)[colnames(res) == "logFC"] <- "log2FC"
  colnames(res)[colnames(res) == "P.Value"] <- "pvalue"
  colnames(res)[colnames(res) == "adj.P.Val"] <- "adj_pvalue"

  res$comparison <- paste0(contrast_feature, ": ", group1, " vs ", group2)
  res$confounders <- paste(confounding_factors, collapse = ", ")

  return(res)
}

In [15]:
adj_pvalue_thr = 0.05
log2FC_thr = 0.5

sex_res <- limma_differential(
  expr = expr,
  sample_info = sample_info,
  contrast_feature = "Sex",
  contrast_direction = "M_vs_F",
  confounding_factors = c("country", "cancer_type", "TP", "toxicity")
)

write_tsv(sex_res, glue("{output}limma_diff_prot-M_vs_F_all.tsv"))

t1_vs_t0_all <- limma_differential(
  expr = expr,
  sample_info = sample_info,
  contrast_feature = "TP",
  contrast_direction = "T1_vs_T0",
  confounding_factors = c("country", "cancer_type", "Sex", "toxicity")
)

write_tsv(t1_vs_t0_all, glue("{output}limma_diff_prot-T1_vs_T0_all.tsv"))

t2_vs_t0_all <- limma_differential(
  expr = expr,
  sample_info = sample_info,
  contrast_feature = "TP",
  contrast_direction = "T2_vs_T0",
  confounding_factors = c("country", "cancer_type", "Sex", "toxicity")
)

write_tsv(t2_vs_t0_all, glue("{output}limma_diff_prot-T2_vs_T0_all.tsv"))

sample_info_t0 = sample_info %>% filter(TP == "T0")
t0_tox_vs_notox <- limma_differential(
  expr = expr,
  sample_info = sample_info_t0,
  contrast_feature = "toxicity",
  contrast_direction = "yes_vs_no",
  confounding_factors = c("country", "cancer_type", "Sex")
)

write_tsv(t0_tox_vs_notox, glue("{output}limma_diff_prot-tox_vs_notox_T0.tsv"))

sample_info_t0_tox = sample_info %>% filter(TP == "T0") %>% filter(toxicity == "yes")
t0_tox_M_vs_F <- limma_differential(
  expr = expr,
  sample_info = sample_info_t0_tox,
  contrast_feature = "Sex",
  contrast_direction = "M_vs_F",
  confounding_factors = c("country", "cancer_type")
)

write_tsv(t0_tox_M_vs_F, glue("{output}limma_diff_prot-M_vs_F_T0_tox.tsv"))

sample_info_t0_notox = sample_info %>% filter(TP == "T0") %>% filter(toxicity == "no")
t0_notox_M_vs_F <- limma_differential(
  expr = expr,
  sample_info = sample_info_t0_notox,
  contrast_feature = "Sex",
  contrast_direction = "M_vs_F",
  confounding_factors = c("country")
)

write_tsv(t0_notox_M_vs_F, glue("{output}limma_diff_prot-M_vs_F_T0_notox.tsv"))

sample_info_M = sample_info %>% filter(Sex == "M")
t2_vs_t0_M <- limma_differential(
  expr = expr,
  sample_info = sample_info_M,
  contrast_feature = "TP",
  contrast_direction = "T2_vs_T0",
  confounding_factors = c("country", "cancer_type", "toxicity")
)

write_tsv(t2_vs_t0_M, glue("{output}limma_diff_prot-T2_vs_T0_M.tsv"))

sample_info_F = sample_info %>% filter(Sex == "F")
t2_vs_t0_F <- limma_differential(
  expr = expr,
  sample_info = sample_info_F,
  contrast_feature = "TP",
  contrast_direction = "T2_vs_T0",
  confounding_factors = c("country", "cancer_type", "toxicity")
)

write_tsv(t2_vs_t0_F, glue("{output}limma_diff_prot-T2_vs_T0_F.tsv"))

sample_info_M = sample_info %>% filter(Sex == "M")
t1_vs_t0_M <- limma_differential(
  expr = expr,
  sample_info = sample_info_M,
  contrast_feature = "TP",
  contrast_direction = "T1_vs_T0",
  confounding_factors = c("country", "cancer_type", "toxicity")
)

write_tsv(t1_vs_t0_M, glue("{output}limma_diff_prot-T1_vs_T0_M.tsv"))

sample_info_F = sample_info %>% filter(Sex == "F")
t1_vs_t0_F <- limma_differential(
  expr = expr,
  sample_info = sample_info_F,
  contrast_feature = "TP",
  contrast_direction = "T1_vs_T0",
  confounding_factors = c("country", "cancer_type", "toxicity")
)

write_tsv(t1_vs_t0_F, glue("{output}limma_diff_prot-T1_vs_T0_F.tsv"))

sample_info_tox = sample_info %>% filter(toxicity == "yes")
tox_t1_vs_t0 <- limma_differential(
  expr = expr,
  sample_info = sample_info_tox,
  contrast_feature = "TP",
  contrast_direction = "T1_vs_T0",
  confounding_factors = c("country", "cancer_type", "Sex")
)

write_tsv(tox_t1_vs_t0, glue("{output}limma_diff_prot-T1_vs_T0_tox.tsv"))

sample_info_notox = sample_info %>% filter(toxicity == "no")
notox_t1_vs_t0 <- limma_differential(
  expr = expr,
  sample_info = sample_info_notox,
  contrast_feature = "TP",
  contrast_direction = "T1_vs_T0",
  confounding_factors = c("country", "Sex")
)

write_tsv(notox_t1_vs_t0, glue("{output}limma_diff_prot-T1_vs_T0_notox.tsv"))

sample_info_M_tox_ids = sample_info %>% filter(Sex == "M") %>% filter(toxicity == "yes") %>% 
pull(`Sample ID`)
sample_info_M_tox = sample_info %>% filter(`Sample ID` %in% sample_info_M_tox_ids)
M_tox_t2_vs_t0 <- limma_differential(
  expr = expr,
  sample_info = sample_info_M_tox,
  contrast_feature = "TP",
  contrast_direction = "T2_vs_T0",
  confounding_factors = c("country", "Sex", "cancer_type")
)

write_tsv(M_tox_t2_vs_t0, glue("{output}limma_diff_prot-T2_vs_T0_M_tox.tsv"))

sample_info_F_tox_ids = sample_info %>% filter(Sex == "F") %>% filter(toxicity == "yes") %>% 
pull(`Sample ID`)
sample_info_F_tox = sample_info %>% filter(`Sample ID` %in% sample_info_F_tox_ids)
F_tox_t2_vs_t0 <- limma_differential(
  expr = expr,
  sample_info = sample_info_F_tox,
  contrast_feature = "TP",
  contrast_direction = "T2_vs_T0",
  confounding_factors = c("country", "Sex", "cancer_type")
)

write_tsv(F_tox_t2_vs_t0, glue("{output}limma_diff_prot-T2_vs_T0_F_tox.tsv"))



Group 1: M
Group 2: F
Contrast: SexM - SexF


Warning message:
"Zero sample variances detected, have been offset away from zero"


Group 1: T1
Group 2: T0
Contrast: TPT1 - TPT0


Warning message:
"Zero sample variances detected, have been offset away from zero"


Group 1: T2
Group 2: T0
Contrast: TPT2 - TPT0


...

Warning message:
"Zero sample variances detected, have been offset away from zero"


Group 1: T2
Group 2: T0
Contrast: TPT2 - TPT0


Warning message:
"Zero sample variances detected, have been offset away from zero"


Group 1: T2
Group 2: T0
Contrast: TPT2 - TPT0


Warning message:
"Zero sample variances detected, have been offset away from zero"
